In [ ]:
# Runs to analyse: $NESYSY_RUNS (set by run.py), else output/full, else output/subset.
# Figures and tables are written to <runs>/paper_results/.
import os
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
RUNS = Path(os.environ.get("NESYSY_RUNS") or next(
    (ROOT / "output" / m for m in ("full", "subset") if (ROOT / "output" / m).exists()),
    ROOT / "output" / "subset",
))
# the paper shows only points with all 25 runs; a subset has fewer runs per point
MIN_RUNS = int(os.environ.get("NESYSY_MIN_RUNS", 25 if "full" in RUNS.parts else 1))
OUT = RUNS / "paper_results"
OUT.mkdir(parents=True, exist_ok=True)
DBS = sorted(RUNS.glob("**/mlflow.db"))


def search_runs(experiment):
    """mlflow.search_runs of an experiment, over all databases below RUNS."""
    frames = []
    for db in DBS:
        mlflow.set_tracking_uri(f"sqlite:///{db}")
        exp = mlflow.get_experiment_by_name(experiment)
        if exp is not None:
            frames.append(mlflow.search_runs(experiment_ids=[exp.experiment_id]))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"runs: {RUNS} ({len(DBS)} database(s)), at least {MIN_RUNS} run(s) per point")


In [ ]:
# Fig. 7: DeepProbLog vs Base under distant supervision (four panels)
PANELS = [
    ("test_5_distant5", "params.cvr", "Clause-to-variable ratio (CVR)", "fig7a"),
    ("test_7_distant", "tags.constraint_weight", "Constraint loss weight", "fig7b"),
    ("test_8_distant", "tags.num_hidden", "Number of layers in the generator network", "fig7c"),
    ("test_9_distant", "tags.num_hidden", "Number of layers in the learner network", "fig7d"),
]

metrics = {
    "metrics.accuracy": "Accuracy",
}

titles = {
    "plain_nn": "Base",
    "plain_nn_sl": "DeepProbLog"
}

for experiment, x_col, x_label, file in PANELS:
    runs = search_runs(experiment)
    if runs.empty:
        print(f"No runs found for {experiment}, skipping.")
        continue
    runs = runs[runs["status"] == "FINISHED"]
    runs = runs.sort_values("start_time").drop_duplicates(subset="tags.mlflow.runName", keep="last").reset_index(drop=True)
    runs[x_col] = runs[x_col].astype(float)

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    axes = [ax]

    for ax, (metric_col, metric_label) in zip(axes, metrics.items()):
        for model_type, group in runs.groupby("tags.model_type"):
            summary = group.groupby(x_col)[metric_col].agg(["mean", "sem"]).sort_index()
            ax.plot(summary.index, summary["mean"], marker="o", label=titles[model_type])
            ax.fill_between(
                summary.index,
                summary["mean"] - summary["sem"],
                summary["mean"] + summary["sem"],
                alpha=0.15,
            )
        ax.set_xlabel(x_label, fontsize=20)
        ax.set_ylabel(metric_label, fontsize=20)
        ax.legend(
            loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=4, fontsize=20
        )
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUT / f"{file}.pdf")
    plt.show()
